In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

FileNotFoundError: [Errno 2] No such file or directory: '/home/talentum/spark/./bin/spark-submit'

# pyspark.sql.functions.to_json

<img src="images/to_json.jpg" align = "left">



Ref -https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.to_json.html

Where `from_json` is used to unpack data, `to_json` does the exact opposite: it takes a structured column (like a `StructType`, an `ArrayType`, or a `MapType`) and packs it into a single JSON string.

This is incredibly useful when you need to send data to a downstream system that expects a specific JSON format, such as pushing records to a **Kafka topic** or saving data to a **NoSQL** database like MongoDB.

# 1. How it Works

The function converts a complex object into a standard JSON string.

1. **Input:** A column of type Struct, Array, or Map.

2. **Logic:** Spark serializes the key-value pairs into a string format.

3. **Output:** A single string column.

# 2. Working Example

Imagine you have a clean table with multiple columns, and you want to bundle the user details into a single JSON string for an API export.

**Step 1:** Create Structured Data  
We'll use struct to group individual columns together first.

In [ ]:
from pyspark.sql.functions import to_json, struct, col

data = [
    (1, "Alice", 30, "New York"),
    (2, "Bob", 25, "Seattle")
]
df = spark.createDataFrame(data, ["id", "name", "age", "city"])

# Group columns into a single Struct named 'user_info'
df_structured = df.select("id", struct("name", "age", "city").alias("user_info"))
# print(df_structured.printSchema())

root
 |-- id: long (nullable = true)
 |-- user_info: struct (nullable = false)
 |    |-- name: string (nullable = true)
 |    |-- age: long (nullable = true)
 |    |-- city: string (nullable = true)

None


**Step 2:** Apply `to_json`

Now, we convert that user_info struct into a JSON string.

In [ ]:
# Convert the Struct column to a JSON String
json_df = df_structured.withColumn("json_string", to_json(col("user_info")))

json_df.select("id", "json_string").show(truncate=False)

+---+-------------------------------------------+
|id |json_string                                |
+---+-------------------------------------------+
|1  |{"name":"Alice","age":30,"city":"New York"}|
|2  |{"name":"Bob","age":25,"city":"Seattle"}   |
+---+-------------------------------------------+



# 3. Key Details to Remember

- **Structs are best:** While you can use to_json on Maps, using struct() allows you to mix different data types (strings, ints, booleans) into the same JSON object.

- **Options:** You can pass a map of options to customize the output, such as {"timestampFormat": "yyyy-MM-dd"} to control how dates look inside the string.

- **Kafka Integration:** If you are writing to Kafka, your DataFrame must have a column named value of type string or binary. `to_json` is the standard way to prepare your data for that value column.

<img src="images/summary_comparison1.jpg" align = "left">